# Wine Quality Prediction: Comparing Random Forest, SGD and SVC
**Oasis Infobyte SIP · Data Analytics · Level 2 · Task 2**

**Objective:** train and compare multiple classification models to predict the quality of wine from its physicochemical properties (acidity, density, alcohol, etc.), handling the strong class imbalance in the quality scores.

**Dataset:** `winequality.csv` - 1,599 red wines with 11 chemical features and a `quality` score (3 to 8) given by tasters.

> The file included here is generated by `generate_data.py` and mimics the UCI / Kaggle *Wine Quality* (red) dataset: same columns, typical ranges, correlations and the same imbalanced score distribution. To use the real data, download `winequality-red.csv` (UCI: archive.ics.uci.edu/dataset/186/wine+quality, or Kaggle), save it as `winequality.csv` (the notebook reads the `;` separator automatically), re-run, and update the written observations to match your results.

**Plan:** load and inspect -> class distribution -> EDA -> correlation heatmap -> imbalance discussion -> bin quality into groups -> stratified split -> train 3 models -> evaluate -> feature importance -> comparison and conclusion.

---
## 1. Load the data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support, f1_score, recall_score)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("images", exist_ok=True)

df = pd.read_csv("winequality.csv")
if df.shape[1] == 1:                      # the original UCI file uses ';' as separator
    df = pd.read_csv("winequality.csv", sep=";")
print("Shape:", df.shape)
df.head()

## 2. Data structure and class distribution of the quality scores

In [ ]:
print("Null values:", int(df.isnull().sum().sum()), "| duplicate rows:", int(df.duplicated().sum()))
print("\nData types:\n", df.dtypes.value_counts())
print("\nDescriptive statistics:")
print(df.describe().T[["mean", "std", "min", "50%", "max"]].round(3))

counts = df["quality"].value_counts().sort_index()
dist = pd.DataFrame({"wines": counts, "percent": (counts / len(df) * 100).round(1)})
print("\nQuality score distribution:\n", dist)
print(f"\nLargest class : smallest class = {counts.max()} : {counts.min()} (about {counts.max() / counts.min():.0f} : 1)")

plt.figure(figsize=(8, 4.5))
bars = plt.bar(counts.index.astype(str), counts.values, color="steelblue")
for b, v in zip(bars, counts.values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v}\n({v / len(df) * 100:.1f}%)", ha="center", va="bottom")
plt.title("Distribution of Wine Quality Scores"); plt.xlabel("Quality score"); plt.ylabel("Number of wines"); plt.ylim(0, counts.max() * 1.18)
plt.savefig("images/01_quality_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** the dataset has **1,599 wines**, 11 numeric chemical features, no missing values and no duplicate rows. Quality scores only range from 3 to 8 and are very unevenly spread: scores **5 (42.6%) and 6 (39.9%)** make up 82.5% of all wines, 7 has 12.4%, while scores **4 (3.3%), 8 (1.1%) and 3 (0.6%, only 10 wines)** are rare. The largest class is about **68 times bigger** than the smallest.

## 3. EDA: distribution of every chemical feature

In [ ]:
features = [c for c in df.columns if c != "quality"]
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(df[col], kde=True, bins=30, ax=ax, color="teal")
    ax.set_title(f"{col} (skew {df[col].skew():.1f})")
    ax.set_xlabel("")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig("images/02_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print(df[features].skew().round(2).sort_values(ascending=False).to_string())

**Observation:** several features are strongly **right-skewed** with long tails of high values: total sulfur dioxide (skew 2.4), free sulfur dioxide (2.1), residual sugar (1.7) and chlorides (1.1). Density, pH and volatile acidity are almost symmetric (skew near 0). Tree-based models such as Random Forest are not affected by skew, while SGD and SVC are sensitive to feature scale, so those two are trained with a `StandardScaler`.

## 4. Correlation heatmap

In [ ]:
corr = df.corr()
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 8})
plt.title("Correlation Heatmap")
plt.savefig("images/03_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Correlation with quality:")
print(corr["quality"].drop("quality").sort_values(ascending=False).round(2).to_string())
print("\nStrongest feature-feature correlations:")
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).drop(columns="quality", index="quality").stack()
print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(5).round(2).to_string())

**Observation:**
- **Alcohol has the strongest positive link to quality (0.53)** and **volatile acidity the strongest negative (-0.41)**. Citric acid (+0.35), sulphates (+0.21) are positive; density (-0.25) and total sulfur dioxide (-0.19) are negative. Residual sugar (-0.04) and pH (-0.06) are almost unrelated to quality.
- Some features are correlated with each other: fixed acidity and pH (-0.69), free and total sulfur dioxide (0.66), fixed acidity and citric acid (0.63). That is expected chemically (more acid means lower pH) and means some features carry overlapping information, which tree-based models handle without trouble.

## 5. Class imbalance: is it a problem?

**Yes.** Scores 3 and 8 have only 10 and 18 wines, and 82.5% of wines are a 5 or a 6. This affects modelling in four ways:
1. **Accuracy becomes misleading:** a model that always answers "5" would already be right 42.6% of the time without learning anything.
2. **Models favour the big classes** and tend to ignore the rare ones, because getting them wrong costs almost nothing in the training loss.
3. **Evaluation is unreliable for rare classes:** a stratified 20% test set would contain only about 2 wines of score 3 and about 4 of score 8, far too few to judge anything.
4. **Neighbouring scores overlap:** the difference between a 5 and a 6 is small and partly subjective, which makes a 6-class problem very noisy.

**Our remedies:** group the scores into fewer classes (next section), use a **stratified** train/test split, set `class_weight="balanced"` so rare classes count more in training, and judge models with **macro-averaged F1** (which treats each class equally) and not accuracy alone.

## 6. Feature engineering: binning quality into groups

We create **3 groups: low (scores 3-5), medium (6) and high (7-8)**.

**Why 3 groups and not 2?**
- Rare scores (3, 4, 8) are merged with their neighbours, which removes the extreme imbalance and gives each class enough samples.
- The groups have a natural business meaning: below average, acceptable and premium wine.
- A binary "good vs bad" split (for example 7 and above as "good") would make only 13.6% of wines positive and would hide the difference between low and medium wines. Three groups keep more information while staying learnable.

In [ ]:
# 3 groups: low (3-5), medium (6), high (7-8)
df["quality_group"] = pd.cut(df["quality"], bins=[0, 5, 6, 10], labels=["low", "medium", "high"])
grp = df["quality_group"].value_counts().reindex(["low", "medium", "high"])
print(pd.DataFrame({"wines": grp, "percent": (grp / len(df) * 100).round(1)}))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
palette = {"low": "#DC2626", "medium": "#F59E0B", "high": "#16A34A"}
raw = df["quality"].value_counts().sort_index()
ax[0].bar(raw.index.astype(str), raw.values, color=[palette["low"] if q <= 5 else palette["medium"] if q == 6 else palette["high"] for q in raw.index])
ax[0].set_title("Original scores coloured by new group"); ax[0].set_xlabel("Quality score"); ax[0].set_ylabel("Wines")
ax[1].bar(grp.index, grp.values, color=[palette[g] for g in grp.index])
for i, v in enumerate(grp.values):
    ax[1].text(i, v, f"{v} ({v / len(df) * 100:.1f}%)", ha="center", va="bottom")
ax[1].set_title("Wines per quality group"); ax[1].set_ylim(0, grp.max() * 1.15)
plt.tight_layout()
plt.savefig("images/04_quality_groups.png", dpi=150, bbox_inches="tight")
plt.show()

# Quick evidence for the decision: same model on the 6 original classes vs the 3 groups
X = df[features]
check = []
for name, target in [("6 original scores", df["quality"]), ("3 groups (low/medium/high)", df["quality_group"].astype(str))]:
    Xa, Xb, ya, yb = train_test_split(X, target, test_size=0.2, random_state=42, stratify=target)
    p = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced").fit(Xa, ya).predict(Xb)
    check.append({"Target": name, "Accuracy": accuracy_score(yb, p), "Macro F1": f1_score(yb, p, average="macro")})
pd.DataFrame(check).set_index("Target").round(3)

**Observation:** the groups are **low 744 wines (46.5%)**, **medium 638 (39.9%)** and **high 217 (13.6%)**. This is much more balanced than the 68:1 ratio before (the high group is still the smallest, which is why we keep class weights). The check at the bottom supports the decision: with the same Random Forest, predicting the 6 original scores gives accuracy 0.559 and **macro F1 of only 0.273**, while predicting the 3 groups gives accuracy 0.641 and **macro F1 of 0.625**, more than twice as good at handling all classes.

## 7. Train / test split with stratification

In [ ]:
y = df["quality_group"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
print("Train proportions:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Test proportions :", y_test.value_counts(normalize=True).round(3).to_dict())

**Observation:** 1,279 wines for training and 320 for testing. Because of `stratify=y`, both sets keep the same class proportions (about 46.5% low, 40% medium, 13.5% high), so the rare "high" group is properly represented in the test set (43 wines).

## 8. Train the three classifiers and evaluate each one

- **Random Forest:** 300 decision trees that vote; handles non-linear relationships and needs no scaling.
- **SGD Classifier (Stochastic Gradient Descent):** a fast linear classifier (default hinge loss, similar to a linear SVM) trained by gradient descent on one sample at a time; needs scaled features.
- **SVC (Support Vector Classifier, RBF kernel):** finds a flexible non-linear boundary between classes; needs scaled features.

All three use `class_weight="balanced"`. For each we show accuracy, the classification report (precision, recall, F1 per class) and the confusion matrix.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "SGD Classifier": make_pipeline(StandardScaler(), SGDClassifier(random_state=42, max_iter=2000, class_weight="balanced")),
    "SVC (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced")),
}
order = ["low", "medium", "high"]
preds = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds[name] = m.predict(X_test)

for name in models:
    print("=" * 62); print(f"{name}  (accuracy = {accuracy_score(y_test, preds[name]):.3f})"); print("=" * 62)
    print(classification_report(y_test, preds[name], labels=order, digits=3))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for a, name in zip(ax, models):
    cm = confusion_matrix(y_test, preds[name], labels=order)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=order, yticklabels=order, cbar=False, ax=a)
    a.set_title(f"{name}\nAccuracy = {accuracy_score(y_test, preds[name]):.3f}"); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("images/05_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:**
- **Accuracy:** Random Forest 64.1%, SVC 63.4% and SGD 60.9%. For comparison, always predicting "low" would score 46.6%, so all three models have learned real patterns.
- **The "medium" group is the hardest** (F1 only 0.49-0.56), because medium wines sit between the other two groups and their chemistry overlaps with both.
- **Errors happen between neighbouring groups.** The models almost never confuse the extremes: a low wine predicted as high happens only 1 time for Random Forest, 4 times for SGD and 1 time for SVC; a high wine predicted as low happens 0, 2 and 0 times.
- **High-quality wines:** SVC finds the most (recall 0.74, 32 of 43) but with lower precision (0.49), meaning more false alarms. Random Forest finds 28 of 43 (recall 0.65) with higher precision (0.53). SGD is the weakest overall.

## 9. Feature importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=features).sort_values()
plt.figure(figsize=(8, 5.5))
plt.barh(imp.index, imp.values, color="seagreen")
plt.title("Random Forest Feature Importance"); plt.xlabel("Importance")
plt.savefig("images/06_rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print(imp.sort_values(ascending=False).round(3).to_string())

**Observation:** **alcohol is the most important feature (0.20), roughly 1.4 times more than the next one**, followed by **volatile acidity (0.15)**. Then come citric acid (0.09), density (0.09) and sulphates (0.09). Free sulfur dioxide (0.06), residual sugar (0.06) and pH (0.06) matter least. This agrees with the correlation analysis: alcohol and volatile acidity (the acid that gives wine a vinegar-like smell) are the two main levers of perceived quality.

## 10. Model comparison table

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, m in models.items():
    p, r, f, _ = precision_recall_fscore_support(y_test, preds[name], average="macro", zero_division=0)
    cv_f1 = cross_val_score(m, X, y, cv=cv, scoring="f1_macro")
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, preds[name]),
                 "Macro precision": p, "Macro recall": r, "Macro F1": f,
                 "Recall of 'high' wines": recall_score(y_test, preds[name], labels=["high"], average=None)[0],
                 "5-fold CV Macro F1": cv_f1.mean()})
comparison = pd.DataFrame(rows).set_index("Model").round(3)

comparison[["Accuracy", "Macro F1", "Recall of 'high' wines"]].plot.bar(figsize=(10, 4.5), rot=0, ylim=(0, 1))
plt.title("Model Comparison (test set)"); plt.legend(loc="upper center", ncol=3)
plt.savefig("images/07_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
comparison

**Observation:**

| Model | Accuracy | Macro F1 | Recall of 'high' wines | 5-fold CV Macro F1 |
|---|---|---|---|---|
| **Random Forest** | **0.641** | **0.625** | 0.651 | **0.617** |
| SVC (RBF) | 0.634 | 0.619 | **0.744** | 0.609 |
| SGD Classifier | 0.609 | 0.583 | 0.628 | 0.582 |

Random Forest is first on accuracy, macro F1 and cross-validated macro F1; SVC is close behind and best at finding premium wines; SGD is last on every measure.

## 11. Conclusion: which model should be deployed?

**Recommendation: Random Forest.**
- It has the **best accuracy (64.1%) and macro F1 (0.625)**, and it also wins in 5-fold cross-validation (0.617), so the result is not luck from a single split.
- It needs **no feature scaling**, copes with skewed and correlated features, and provides **feature importance**, so winemakers can see *why* a wine is rated low or high (mostly alcohol and volatile acidity).
- **When to choose SVC instead:** if the priority is to catch as many high-quality wines as possible (recall 0.74) and extra false alarms are acceptable. **SGD** is only worth considering if training speed on a very large dataset matters more than accuracy.

**Caveats:**
- The gap between Random Forest and SVC is small (about 0.6 points of macro F1 on only 320 test wines), so they are practically close.
- Accuracy of about 64% is moderate: taste is subjective and chemical measurements cannot capture everything, so the model is best used as a **screening aid** that supports human tasters, not replaces them.
- The medium group stays hard to separate. Next steps: tune hyperparameters (grid search), try gradient boosting, add more data, and consider oversampling techniques such as SMOTE.
- The data used here is synthetic and built to mimic the real dataset, so results on the real UCI file will differ somewhat.